<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 7 → Classificação e Regressão Supervisionada</div>

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Gerando os Dados e Criando o DataFrame</b></span>

In [ ]:
def definir_estado(linha):
    if linha["temperatura"] > 40 or linha["vibracao"] > 5 or linha["corrente"] > 15:
        return "Falha"
    if linha["temperatura"] > 34 or linha["vibracao"] > 3.8 or linha["corrente"] > 12:
        return "Alerta"
    return "Normal"


# GERANDO OSDADOS
np.random.seed(42)
n = 300

dados = pd.DataFrame({
    "temperatura": np.random.normal(30, 8, n),
    "vibracao": np.random.normal(3, 1.5, n),
    "corrente": np.random.normal(10, 3, n)
})

dados["estado"] = dados.apply(definir_estado, axis=1)


# GRÁFICO DOS ESTADOS
for estado in ["Normal", "Alerta", "Falha"]:
    grupo = dados[dados["estado"] == estado]
    plt.scatter(grupo["temperatura"], grupo["vibracao"], label=estado)

plt.title("Estados do equipamento")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vibração")
plt.legend()
plt.grid()
plt.show()


# TREINO E MODELO
X = dados[["temperatura", "vibracao", "corrente"]]
y = dados["estado"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_treino, y_treino)

previsao = modelo.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, previsao) * 100:.2f}%")


# MATRIZ DE CONFUSÃO
classes = modelo.classes_
matriz = confusion_matrix(y_teste, previsao, labels=classes)

plt.imshow(matriz)
plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão")

for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, matriz[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


# IMPORTÂNCIA DOS SENSORES
pd.Series(
    modelo.feature_importances_,
    index=X.columns
).plot(kind="barh")

plt.title("Importância dos Sensores")
plt.xlabel("Importância")
plt.show()


# NOVA LEITURA
nova_leitura = pd.DataFrame({
    "temperatura": [42],
    "vibracao": [5.4],
    "corrente": [14]
})

print("Estado previsto:", modelo.predict(nova_leitura)[0])

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
# GERANDO OS DADOS
np.random.seed(42)
n = 250

dados = pd.DataFrame({
    "temperatura": np.random.normal(30, 6, n),
    "vibracao": np.random.normal(3, 1, n),
    "carga": np.random.uniform(20, 100, n)
})

dados["consumo_kwh"] = (
    0.35 * dados["temperatura"]
    + 1.8 * dados["vibracao"]
    + 0.22 * dados["carga"]
    + np.random.normal(0, 2, n)
)

print(dados.head())


# CARGA x CONSUMO
plt.figure(figsize=(8, 5))
plt.scatter(dados["carga"], dados["consumo_kwh"])
plt.title("Carga x Consumo de Energia")
plt.xlabel("Carga da máquina (%)")
plt.ylabel("Consumo de energia (kWh)")
plt.grid()
plt.show()


# TREINO E MODELO
X = dados[["temperatura", "vibracao", "carga"]]
y = dados["consumo_kwh"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

previsoes = modelo.predict(X_teste)


# MÉTRICAS
mae = mean_absolute_error(y_teste, previsoes)
rmse = np.sqrt(mean_squared_error(y_teste, previsoes))
r2 = r2_score(y_teste, previsoes)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")


# REAL x PREVISTO
plt.figure(figsize=(7, 5))
plt.scatter(y_teste, previsoes)

limites = [
    min(y_teste.min(), previsoes.min()),
    max(y_teste.max(), previsoes.max())
]

plt.plot(limites, limites, "--")
plt.title("Real x Previsto")
plt.xlabel("Consumo real (kWh)")
plt.ylabel("Consumo previsto (kWh)")
plt.grid()
plt.show()


# RESÍDUOS
residuos = y_teste - previsoes

plt.figure(figsize=(7, 5))
plt.scatter(previsoes, residuos)
plt.axhline(0, linestyle="--")
plt.title("Análise dos Resíduos")
plt.xlabel("Consumo previsto")
plt.ylabel("Erro")
plt.grid()
plt.show()


# NOVA LEITURA
nova_leitura = pd.DataFrame({
    "temperatura": [35],
    "vibracao": [4],
    "carga": [80]
})

resultado = modelo.predict(nova_leitura)

print(f"Consumo previsto: {resultado[0]:.2f} kWh")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
def calcular_metricas(real, previsto):
    real = np.array(real)
    previsto = np.array(previsto)

    mae = mean_absolute_error(real, previsto)
    mse = mean_squared_error(real, previsto)
    rmse = np.sqrt(mse)
    r2 = r2_score(real, previsto)
    mape = np.mean(np.abs((real - previsto) / real)) * 100

    print(f"MAE: {mae:.2f}")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.2f}")
    print(f"MAPE: {mape:.2f}%")


real = [20, 25, 30, 35, 40]
previsto = [22, 24, 27, 36, 38]

calcular_metricas(real, previsto)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# GERANDO OS DADOS
np.random.seed(42)
n = 500
t = np.arange(n)

dados = pd.DataFrame({
    "tempo": pd.date_range("2026-01-01", periods=n, freq="h")
})

dados["carga"] = 60 + 15*np.sin(t/20) + np.random.normal(0, 4, n)
dados["vibracao"] = 3 + 0.02*dados["carga"] + np.random.normal(0, 0.3, n)
dados["temperatura"] = (
    25 + 0.18*dados["carga"] + 0.8*dados["vibracao"]
    + 0.01*t + np.random.normal(0, 1, n)
)

dados["temperatura_proxima"] = dados["temperatura"].shift(-1)
dados = dados.dropna()


# TREINO E TESTE
corte = int(len(dados) * 0.8)
treino, teste = dados.iloc[:corte], dados.iloc[corte:]

colunas = ["carga", "vibracao", "temperatura"]

X_treino = treino[colunas]
y_treino = treino["temperatura_proxima"]

X_teste = teste[colunas]
y_teste = teste["temperatura_proxima"]


# MODELO
modelo = Pipeline([
    ("escala", StandardScaler()),
    ("regressao", LinearRegression())
])

modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)


# MÉTRICAS
mae = mean_absolute_error(y_teste, previsoes)
rmse = np.sqrt(mean_squared_error(y_teste, previsoes))
r2 = r2_score(y_teste, previsoes)

print(f"MAE: {mae:.2f} °C")
print(f"RMSE: {rmse:.2f} °C")
print(f"R²: {r2:.2f}")


# FUNÇÃO DE GRÁFICO
def grafico(x, y1, y2, label1, label2, titulo):
    plt.figure(figsize=(10, 4))
    plt.plot(x, y1, label=label1)
    plt.plot(x, y2, label=label2)
    plt.title(titulo)
    plt.xlabel("Tempo")
    plt.ylabel("Temperatura (°C)")
    plt.legend()
    plt.grid()
    plt.show()


# GRÁFICO 1 - TREINO x TESTE
grafico(
    dados["tempo"],
    dados["temperatura"],
    np.where(
        dados.index < corte,
        dados["temperatura"],
        np.nan
    ),
    "Temperatura",
    "Treino",
    "Treino antes, teste depois"
)

plt.figure(figsize=(10, 4))
plt.plot(treino["tempo"], treino["temperatura"], label="Treino")
plt.plot(teste["tempo"], teste["temperatura"], label="Teste")
plt.axvline(teste["tempo"].iloc[0], linestyle="--")
plt.title("Treino antes, teste depois")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid()
plt.show()


# GRÁFICO 2 - REAL x PREVISTO
grafico(
    teste["tempo"],
    y_teste,
    previsoes,
    "Real",
    "Previsto",
    "Temperatura real x prevista"
)


# GRÁFICO 3 - RESÍDUOS
residuos = y_teste.values - previsoes

plt.figure(figsize=(8, 4))
plt.scatter(previsoes, residuos)
plt.axhline(0, linestyle="--")
plt.title("Erros do modelo")
plt.xlabel("Temperatura prevista")
plt.ylabel("Erro")
plt.grid()
plt.show()


# NOVA LEITURA
nova_leitura = pd.DataFrame({
    "carga": [72],
    "vibracao": [4.3],
    "temperatura": [41]
})

resultado = modelo.predict(nova_leitura)

print(f"Temperatura prevista: {resultado[0]:.2f} °C")